# Hyperparameter Tuning — Logistic Regression + Class Weights

**Objective:** Tune best model from baseline comparison to maximize Balanced Accuracy and Recall.

**Model:** Logistic Regression + Class Weights  
**Technique:** RandomizedSearchCV with 5-fold Cross Validation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score,
    recall_score,
    f1_score,
    precision_score,
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import pickle
import os

print("✅ Libraries loaded")

✅ Libraries loaded


## Load Data

In [3]:
# Use original imbalanced data — class_weight handles balancing
X_train = pd.read_csv("../data/processed/X_train_original.csv")
y_train = pd.read_csv("../data/processed/y_train_original.csv").values.ravel()
X_test  = pd.read_csv("../data/processed/X_test.csv")
y_test  = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

X_train: (8000, 25)
X_test:  (2000, 25)


## 1. Baseline Performance (Before Tuning)
Establish baseline to compare against tuned model

In [4]:
baseline = LogisticRegression(
    class_weight="balanced",
    random_state=42,
    max_iter=1000
)
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
y_prob_base = baseline.predict_proba(X_test)[:, 1]

print("📊 Baseline Performance (Default Parameters):")
print(f"  Balanced Accuracy → {balanced_accuracy_score(y_test, y_pred_base)*100:.2f}%")
print(f"  Recall            → {recall_score(y_test, y_pred_base)*100:.2f}%")
print(f"  Precision         → {precision_score(y_test, y_pred_base)*100:.2f}%")
print(f"  F1 Score          → {f1_score(y_test, y_pred_base)*100:.2f}%")
print(f"  ROC-AUC           → {roc_auc_score(y_test, y_prob_base)*100:.2f}%")

📊 Baseline Performance (Default Parameters):
  Balanced Accuracy → 53.96%
  Recall            → 53.91%
  Precision         → 7.42%
  F1 Score          → 13.04%
  ROC-AUC           → 53.07%


## 2. Define Hyperparameter Search Space

**Key parameters for Logistic Regression:**
- `C` → Regularization strength (smaller = more regularization)
- `solver` → Optimization algorithm
- `penalty` → Type of regularization (L1, L2, ElasticNet)
- `max_iter` → Maximum iterations for convergence
- `l1_ratio` → Mix ratio for ElasticNet penalty

In [5]:
param_dist = {
    "C":          [0.001, 0.01, 0.1, 0.5, 1, 5, 10, 50, 100],
    "solver":     ["saga", "liblinear"],
    "penalty":    ["l1", "l2"],
    "max_iter":   [500, 1000, 2000, 5000],
    "tol":        [1e-4, 1e-3, 1e-2]
}

print("🔍 Search Space:")
for param, values in param_dist.items():
    print(f"  {param}: {values}")

total = 1
for v in param_dist.values():
    total *= len(v)
print(f"\nTotal combinations: {total}")
print(f"RandomizedSearchCV will test: 50 random combinations")

🔍 Search Space:
  C: [0.001, 0.01, 0.1, 0.5, 1, 5, 10, 50, 100]
  solver: ['saga', 'liblinear']
  penalty: ['l1', 'l2']
  max_iter: [500, 1000, 2000, 5000]
  tol: [0.0001, 0.001, 0.01]

Total combinations: 432
RandomizedSearchCV will test: 50 random combinations


## 3. RandomizedSearchCV

**Why RandomizedSearchCV over GridSearchCV:**
- GridSearch tests ALL combinations → too slow
- RandomizedSearch tests random subset → faster, good enough
- With 5-fold CV → each combination trained 5 times
- Scoring on Balanced Accuracy → most honest metric for imbalanced data

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=LogisticRegression(
        class_weight="balanced",
        random_state=42
    ),
    param_distributions=param_dist,
    n_iter=70,                        # test 70 random combinations
    scoring="balanced_accuracy",      # optimize for balanced accuracy
    cv=cv,                            # 5 fold cross validation
    n_jobs=-1,                        # use all CPU cores
    random_state=42,
    verbose=1
)

print("⚙️ Running RandomizedSearchCV...")
print("This may take a minute...\n")

search.fit(X_train, y_train)

print(f"\n✅ Best Parameters Found:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\n  CV Balanced Accuracy: {search.best_score_*100:.2f}%")

⚙️ Running RandomizedSearchCV...
This may take a minute...

Fitting 5 folds for each of 70 candidates, totalling 350 fits


/Users/shamilikolli/Desktop/return_prediction/venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/shamilikolli/Desktop/return_prediction/venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/shamilikolli/Desktop/return_prediction/venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/shamilikolli/Desktop/return_prediction/venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/shamilikolli/Desktop/return_prediction/venv/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The ma


✅ Best Parameters Found:
  tol: 0.0001
  solver: liblinear
  penalty: l2
  max_iter: 500
  C: 0.001

  CV Balanced Accuracy: 49.99%


In [7]:
best_model = search.best_estimator_

y_pred_tuned = best_model.predict(X_test)
y_prob_tuned = best_model.predict_proba(X_test)[:, 1]

print("📊 Performance Comparison:\n")
print(f"{'Metric':<22} {'Baseline':>10} {'Tuned':>10} {'Change':>10}")
print("-" * 55)

metrics = {
    "Balanced Accuracy": (
        balanced_accuracy_score(y_test, y_pred_base)*100,
        balanced_accuracy_score(y_test, y_pred_tuned)*100
    ),
    "Recall": (
        recall_score(y_test, y_pred_base)*100,
        recall_score(y_test, y_pred_tuned)*100
    ),
    "Precision": (
        precision_score(y_test, y_pred_base)*100,
        precision_score(y_test, y_pred_tuned)*100
    ),
    "F1 Score": (
        f1_score(y_test, y_pred_base)*100,
        f1_score(y_test, y_pred_tuned)*100
    ),
    "ROC-AUC": (
        roc_auc_score(y_test, y_prob_base)*100,
        roc_auc_score(y_test, y_prob_tuned)*100
    )
}

for metric, (base, tuned) in metrics.items():
    change = tuned - base
    arrow = "↑" if change > 0 else "↓"
    print(f"{metric:<22} {base:>9.2f}% {tuned:>9.2f}% {arrow}{abs(change):>7.2f}%")

📊 Performance Comparison:

Metric                   Baseline      Tuned     Change
-------------------------------------------------------
Balanced Accuracy          53.96%     53.35% ↓   0.61%
Recall                     53.91%     51.56% ↓   2.34%
Precision                   7.42%      7.28% ↓   0.13%
F1 Score                   13.04%     12.77% ↓   0.28%
ROC-AUC                    53.07%     53.03% ↓   0.04%
